In [56]:
!pip install flask pyngrok opencv-python pillow numpy scikit-learn xgboost timm torch torchvision tensorflow


In [57]:
!pip install timm scikit-image torch torchvision tensorflow


In [58]:
!mkdir templates
!mkdir static

!mkdir static/uploads

mkdir: cannot create directory ‘templates’: File exists
mkdir: cannot create directory ‘static’: File exists
mkdir: cannot create directory ‘static/uploads’: File exists


In [59]:
from google.colab import drive    # connects drive to this Colab
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [84]:
%%writefile app.py

from flask import Flask, render_template, request
import os
import cv2
import numpy as np
import joblib
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ======== DEEP MODELS ========
import torch
import timm
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from skimage.feature import graycomatrix, graycoprops

CANCER_INFO = {

"Melanoma":
"A serious form of skin cancer that develops from pigment-producing cells.",

"Basal Cell Carcinoma":
"The most common but least aggressive skin cancer.",

"Actinic Keratosis":
"A precancerous skin condition caused by long-term sun exposure.",

"Benign Keratosis":
"A non-cancerous skin lesion that can resemble melanoma.",

"Dermatofibroma":
"A benign skin tumor that appears as a firm bump on the skin.",

"Vascular Lesion":
"Skin abnormalities caused by irregular blood vessels.",

"Melanocytic Nevus":
"A benign mole that usually does not require treatment."
}

# =====================================================
# LOAD TRAINED MODELS
# =====================================================

xgb = joblib.load("/content/drive/My Drive/finalModels/xgb_skin_cancer_model.pkl")
scaler = joblib.load("/content/drive/My Drive/finalModels/scaler.pkl")
pca = joblib.load("/content/drive/My Drive/finalModels/pca.pkl")
le = joblib.load("/content/drive/My Drive/finalModels/label_encoder.pkl")

explainer = shap.TreeExplainer(xgb)

IMG_SIZE = 192

app = Flask(__name__)
# simple user storage (for demo)
users = {
    "admin": "1234"
}
UPLOAD_FOLDER = "static/uploads"
app.config["UPLOAD_FOLDER"] = UPLOAD_FOLDER


# =====================================================
# LOAD FEATURE EXTRACTORS
# =====================================================

cnn_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    pooling='avg',
    input_shape=(192,192,3)
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

swin = timm.create_model(
    'swin_tiny_patch4_window7_224',
    pretrained=True,
    num_classes=0
).to(device)

swin.eval()


# =====================================================
# SCORE-CAM MODEL
# =====================================================

num_classes = len(le.classes_)

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = GlobalAveragePooling2D()(base_model.output)
output = Dense(num_classes, activation="softmax")(x)

scorecam_model = Model(inputs=base_model.input, outputs=output)


# =====================================================
# HANDCRAFTED FEATURES
# =====================================================

def handcrafted(img):

    gray = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_BGR2GRAY)

    glcm = graycomatrix(
        gray,
        [1],
        [0,np.pi/4,np.pi/2,3*np.pi/4],
        256,
        symmetric=True
    )

    contrast = graycoprops(glcm,'contrast').flatten()
    energy = graycoprops(glcm,'energy').flatten()
    homogeneity = graycoprops(glcm,'homogeneity').flatten()
    correlation = graycoprops(glcm,'correlation').flatten()

    return np.hstack([contrast,energy,homogeneity,correlation])


# =====================================================
# HYBRID FEATURE EXTRACTION
# =====================================================

def extract_features(img):

    img = cv2.resize(img,(192,192))
    img = img / 255.0

    cnn_feat = cnn_model.predict(img[np.newaxis,...], verbose=0)

    img_224 = cv2.resize(img,(224,224))

    tensor = torch.tensor(img_224)\
        .permute(2,0,1)\
        .unsqueeze(0)\
        .float()\
        .to(device)

    with torch.no_grad():
        swin_feat = swin(tensor).cpu().numpy()

    hc_feat = handcrafted(img).reshape(1,-1)

    fused = np.concatenate([cnn_feat, swin_feat, hc_feat], axis=1)

    return fused


# =====================================================
# SCORE-CAM FUNCTION
# =====================================================

def compute_scorecam(model, img_array, class_idx, layer_name="block6a_expand_conv"):

    cam_model = Model(
        inputs=model.input,
        outputs=[model.get_layer(layer_name).output, model.output]
    )

    conv_outputs, _ = cam_model(img_array)

    conv_outputs = conv_outputs[0].numpy()
    H, W, C = conv_outputs.shape

    scorecam = np.zeros((H, W), dtype=np.float32)

    for i in range(C):

        fmap = conv_outputs[:,:,i]
        fmap = np.maximum(fmap,0)

        if np.max(fmap)==0:
            continue

        fmap_norm = fmap/np.max(fmap)
        fmap_resized = cv2.resize(fmap_norm,(IMG_SIZE,IMG_SIZE))

        masked_img = img_array[0]*fmap_resized[...,np.newaxis]
        masked_img = np.expand_dims(masked_img,axis=0)

        score = model.predict(masked_img,verbose=0)[0][class_idx]

        scorecam += score*fmap_norm

    scorecam = np.maximum(scorecam,0)

    if np.max(scorecam)!=0:
        scorecam/=np.max(scorecam)

    return scorecam


# =====================================================
# PREDICTION FUNCTION
# =====================================================

def predict_image(img_path):

    img = cv2.imread(img_path)

    fused = extract_features(img)
    fused_scaled = scaler.transform(fused)
    fused_pca = pca.transform(fused_scaled)

    pred = xgb.predict(fused_pca)
    probs = xgb.predict_proba(fused_pca)

    label = le.inverse_transform(pred)[0]

    class_map = {
        "nv":"Melanocytic Nevus",
        "mel":"Melanoma",
        "bkl":"Benign Keratosis",
        "bcc":"Basal Cell Carcinoma",
        "akiec":"Actinic Keratosis",
        "vasc":"Vascular Lesion",
        "df":"Dermatofibroma"
    }

    label = class_map.get(label,label)

    # ⭐ SCORE-CAM
    img_resized = cv2.resize(img,(IMG_SIZE,IMG_SIZE))/255.0
    img_input = np.expand_dims(img_resized,axis=0)

    pred_class = int(pred[0])

    scorecam = compute_scorecam(
        scorecam_model,
        img_input,
        pred_class
    )

    heatmap_resized = cv2.resize(scorecam,(IMG_SIZE,IMG_SIZE))

    heatmap_color = cv2.applyColorMap(
        np.uint8(255*heatmap_resized),
        cv2.COLORMAP_JET
    )

    overlay = cv2.addWeighted(
        cv2.resize(img,(IMG_SIZE,IMG_SIZE)),
        0.6,
        heatmap_color,
        0.4,
        0
    )

    overlay = overlay.astype(np.uint8)

    cam_path = os.path.join(UPLOAD_FOLDER,"scorecam.jpg")
    cv2.imwrite(cam_path,overlay)

    # ⭐ SHAP
    shap_values = explainer.shap_values(fused_pca)

    plt.figure(figsize=(6,4))

   # ✅ CLEAN FEATURE IMPORTANCE (NO INTERACTIONS)
    shap.summary_plot(
    shap_values,
    fused_pca,
    plot_type="bar",
    show=False
)

    shap_path = os.path.join(UPLOAD_FOLDER,"shap.png")
    plt.savefig(shap_path,bbox_inches='tight')
    plt.close()

    return label, cam_path, shap_path


# =====================================================
# ROUTES
# =====================================================


@app.route("/", methods=["GET","POST"])
def login():

    error = None

    if request.method == "POST":

        username = request.form["username"]
        password = request.form["password"]

        if username in users and users[username] == password:
            return render_template("home.html")
        else:
            error = "Invalid username or password"

    return render_template("login.html", error=error)

@app.route("/signup", methods=["GET","POST"])
def signup():

    message = None

    if request.method == "POST":

        username = request.form["username"]
        password = request.form["password"]

        if username in users:
            message = "User already exists"
        else:
            users[username] = password
            message = "Account created successfully"

    return render_template("signup.html", message=message)


@app.route("/home")
def home():
    return render_template("home.html")


@app.route("/predict", methods=["GET","POST"])
def predict():

    prediction=None
    img_path=None
    cam_path=None
    shap_path=None
    description=None
    cancer=False

    if request.method=="POST":

        file=request.files["file"]

        if file:

            save_path=os.path.join(app.config["UPLOAD_FOLDER"],file.filename)
            file.save(save_path)

            prediction,cam_path,shap_path=predict_image(save_path)

            img_path=save_path

            # get description
            description = CANCER_INFO.get(prediction,"")

            # detect cancer
            if prediction!="Melanocytic Nevus":
                cancer=True

    return render_template(
        "predict.html",
        prediction=prediction,
        description=description,
        img_path=img_path,
        cam_path=cam_path,
        shap_path=shap_path,
        cancer=cancer
    )

@app.route("/precautions")
def precautions():
    return render_template("precautions.html")


@app.route("/about")
def about():
    return render_template("about.html")


if __name__=="__main__":
    app.run(port=5000,host="0.0.0.0")


Overwriting app.py


In [85]:
%%writefile templates/login.html

<!DOCTYPE html>
<html lang="en">

<head>
<meta charset="UTF-8">
<title>Skin Cancer Detection - Login</title>

<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">

</head>

<body class="login-body">

<div class="login-container">

<h2>Login</h2>

<form method="POST">

<input type="text" name="username" placeholder="Username" required>

<input type="password" name="password" placeholder="Password" required>

<button type="submit">Login</button>

</form>

</div>

</body>
</html>

Overwriting templates/login.html


In [86]:
%%writefile templates/signup.html

<!DOCTYPE html>
<html>
<head>
<title>FusionDerm Sign Up</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>

<body>

<div class="login-card">

<h1>Create Account</h1>

<form method="POST">

<input type="text" name="username" placeholder="Username" required>

<input type="password" name="password" placeholder="Password" required>

<button type="submit">Sign Up</button>

</form>

{% if message %}
<p style="color:green">{{ message }}</p>
{% endif %}

<a href="/">Back to Login</a>

</div>

</body>
</html>

Overwriting templates/signup.html


In [87]:
%%writefile templates/home.html

<!DOCTYPE html>
<html>
<head>

<title>FusionDerm - Skin Cancer Awareness</title>

<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">

</head>

<body class="page-bg">

<nav class="navbar">

<div class="logo">FusionDerm</div>

<div class="nav-links">
<a href="/home">Home</a>
<a href="/predict">Prediction</a>
<a href="/about">About</a>
</div>

</nav>


<!-- Skin Cancer Awareness -->

<section class="awareness">

<h1>Skin Cancer Awareness</h1>

<img src="https://media.istockphoto.com/id/1927028123/photo/beauty-photo-of-woman-with-clean-and-healthy-skin-showing-her-face.jpg?s=612x612&w=0&k=20&c=zP6RKP_gP1bKgO9lt3hGcRkEzQKXo96ixL5WixseUJ4=" class="awareness-img">

<p>

Skin cancer is one of the most common cancers worldwide. It occurs when skin cells grow abnormally due to damage caused by ultraviolet (UV) radiation from the sun. Early detection and treatment significantly increase survival rates.

FusionDerm is an AI-assisted system that helps detect different skin cancer types using dermoscopic image analysis.

</p>

</section>

<section class="types">

<h1>Types of Skin Cancer</h1>


<!-- Melanoma -->
<div class="section">
<div class="card-row">

<img src="https://upload.wikimedia.org/wikipedia/commons/6/6c/Melanoma.jpg">

<div class="card-text">
<h3>Melanoma (Mel)</h3>

<p>
Melanoma is the most aggressive type of skin cancer. It develops in melanocytes and can spread rapidly if untreated.
</p>

<h4>Remedies</h4>

<ul>
<li>Early surgical removal</li>
<li>Immunotherapy</li>
<li>Regular dermatological checkups</li>
</ul>

</div>
</div>
</div>



<!-- BCC -->
<div class="section">
<div class="card-row reverse">

<img src="https://upload.wikimedia.org/wikipedia/commons/1/1a/Superficial_basal_cell_carcinoma.jpg">

<div class="card-text">

<h3>Basal Cell Carcinoma (BCC)</h3>

<p>
BCC is the most common form of skin cancer and usually appears as a shiny bump on sun-exposed skin.
</p>

<h4>Remedies</h4>

<ul>
<li>Surgical removal</li>
<li>Cryotherapy</li>
<li>Radiation therapy</li>
</ul>

</div>
</div>
</div>



<!-- AK -->
<div class="section">
<div class="card-row">

<img src="https://sccsnj.com/wp-content/uploads/2020/02/pre-cancerous-condition-e1582238981182.png">

<div class="card-text">

<h3>Actinic Keratosis (AK)</h3>

<p>
Actinic Keratosis is a precancerous skin condition caused by long-term sun exposure.
</p>

<h4>Remedies</h4>

<ul>
<li>Topical medications</li>
<li>Cryotherapy</li>
<li>Laser therapy</li>
</ul>

</div>
</div>
</div>



<!-- BKL -->
<div class="section">
<div class="card-row reverse">

<img src="https://www.rodeoderm.com/wp-content/uploads/Seborrheic-Keratosis.jpg">

<div class="card-text">

<h3>Benign Keratosis (BKL)</h3>

<p>
Benign Keratosis is a non-cancerous skin lesion that may resemble melanoma but is harmless.
</p>

<h4>Remedies</h4>

<ul>
<li>Dermatologist monitoring</li>
<li>Laser removal if needed</li>
</ul>

</div>
</div>
</div>



<!-- DF -->
<div class="section">
<div class="card-row">

<img src="https://perridermatology.com/wp-content/uploads/2015/09/DF1_800_600_http-www.pcds_.org_.ukeeassetsimgwatermark.gif_0_0_80_r_b_-5_-5_.jpg">

<div class="card-text">

<h3>Dermatofibroma (DF)</h3>

<p>
Dermatofibroma is a benign skin tumor that appears as a firm bump on the skin.
</p>

<h4>Remedies</h4>

<ul>
<li>No treatment required</li>
<li>Surgical removal if uncomfortable</li>
</ul>

</div>
</div>
</div>



<!-- VASC -->
<div class="section">
<div class="card-row reverse">

<img src="https://www.mydcsi.com/wp-content/uploads/2025/04/What_Causes_Vascular_Skin_Lesions1-1080x675.png">

<div class="card-text">

<h3>Vascular Lesion (Vasc)</h3>

<p>
Vascular lesions are skin abnormalities caused by irregular blood vessels.
</p>

<h4>Remedies</h4>

<ul>
<li>Laser therapy</li>
<li>Dermatological monitoring</li>
</ul>

</div>
</div>
</div>

</section>

Overwriting templates/home.html


In [88]:
%%writefile templates/predict.html

<!DOCTYPE html>
<html>
<head>

<title>Prediction</title>

<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">

</head>

<body class="page-bg">

<nav class="navbar">

<div class="logo">FusionDerm</div>

<div class="nav-links">
<a href="/home">Home</a>
<a href="/predict">Prediction</a>
<a href="/about">About</a>
</div>

</nav>


<div class="container">

<h2>Upload Dermoscopic Image</h2>

<form method="POST" enctype="multipart/form-data">

<input type="file" name="file" required onchange="previewImage(event)">
<button type="submit">Predict</button>

</form>

<div id="loader" class="loader"></div>

<p id="loading-text" class="loading-text">
Analyzing image with AI model...
</p>
<img id="preview" class="preview-img">


{% if img_path %}
<img src="{{ img_path }}" class="preview">
{% endif %}


{% if prediction %}

<div class="result-box">

<h2>Prediction : {{ prediction }}</h2>

<p class="cancer-desc">
{{ description }}
</p>


<h2>AI Explanation</h2>

{% if cam_path %}

<h3>Heatmap (ScoreCAM)</h3>

<img src="{{ cam_path }}" class="explain-img">

<p class="ai-text">
ScoreCAM highlights the regions of the skin lesion that influenced the model's prediction.
Red areas indicate the most important regions used by the AI system during classification.
</p>

{% endif %}


{% if shap_path %}

<h3>Feature Importance (SHAP)</h3>

<img src="{{ shap_path }}" class="shap-img">

<p class="ai-text">
SHAP explains which extracted features contributed most to the final prediction.
It helps understand how the hybrid AI model made its decision.
</p>

{% endif %}


{% if cancer %}

<a href="/precautions" class="warning-btn">
View Precautions
</a>

{% endif %}


<div class="disclaimer">

⚠ Disclaimer:

This system is intended for educational and research purposes only.
It does not replace professional medical diagnosis.
Consult a dermatologist for medical advice.

</div>


</div>

{% endif %}

</div>

<script>

<script>

function previewImage(event){

const preview = document.getElementById("preview");

preview.src = URL.createObjectURL(event.target.files[0]);

preview.style.display = "block";

}

// show loader when prediction starts
document.querySelector("form").onsubmit = function(){

document.getElementById("loader").style.display = "block";

document.getElementById("loading-text").style.display = "block";

}

</script>

</script>
</body>
</html>

Overwriting templates/predict.html


In [89]:
%%writefile templates/precautions.html

<!DOCTYPE html>
<html>
<head>

<title>Precautions</title>

<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">

</head>

<body class="precautions-body">


<div class="precautions-container">

<h1>Skin Cancer Precautions</h1>

<ul>

<li>Consult a dermatologist immediately.</li>

<li>Avoid excessive sun exposure.</li>

<li>Use sunscreen SPF 30 or higher.</li>

<li>Wear protective clothing.</li>

<li>Regularly monitor skin changes.</li>

</ul>

<a href="/home" class="start-btn">Back to Home</a>

</div>


</body>
</html>

Overwriting templates/precautions.html


In [90]:
%%writefile templates/about.html
<!DOCTYPE html>
<html lang="en">

<head>

<meta charset="UTF-8">
<title>About - Skin Cancer Detection System</title>

<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">

</head>

<body class="page-bg">

<div class="about-section">

<h1>Skin Cancer Detection System</h1>
<p><b>Intelligent AI System for Early Skin Cancer Detection</b></p>


<h2>The Problem</h2>

<p>
Skin cancer is one of the most common and dangerous forms of cancer worldwide.
Early diagnosis is critical for effective treatment, but traditional detection
relies on manual examination by dermatologists. This process can be time-consuming
and may vary depending on medical expertise. An automated and reliable diagnostic
support system can help improve early detection and assist healthcare professionals.
</p>


<h2>Our Solution</h2>

<p>
This project introduces a <b>Hybrid AI-Based Skin Cancer Classification System</b>
that automatically analyzes dermoscopic skin lesion images and predicts the type
of skin cancer. The system combines deep learning, transformer models, and
handcrafted image features to extract meaningful patterns from skin lesions and
provide accurate predictions.
</p>


<h2>How the System Works</h2>

<ol>

<li><b>Image Upload</b><br>
Users upload a dermoscopic skin lesion image through the web interface.</li>

<li><b>Image Preprocessing</b><br>
The image is resized and normalized to ensure consistent input for the model.</li>

<li><b>Feature Extraction</b>
<ul>
<li>EfficientNet (CNN) extracts local image patterns such as texture and color variations.</li>
<li>Swin Transformer captures global contextual relationships in the lesion.</li>
<li>GLCM Handcrafted Features analyze statistical texture characteristics.</li>
</ul>
</li>

<li><b>Feature Fusion</b><br>
All extracted features are combined into a unified hybrid feature representation.</li>

<li><b>Dimensionality Reduction</b><br>
Principal Component Analysis (PCA) reduces redundant information and improves computational efficiency.</li>

<li><b>Classification</b><br>
The optimized features are classified using XGBoost, which predicts the skin lesion category.</li>

</ol>


<h2>Technologies Used</h2>

<ul>

<li><b>Programming Language</b>
<ul>
<li>Python</li>
</ul>
</li>

<li><b>Deep Learning</b>
<ul>
<li>TensorFlow</li>
<li>PyTorch</li>
<li>Keras</li>
</ul>
</li>

<li><b>Machine Learning</b>
<ul>
<li>Scikit-learn</li>
<li>XGBoost</li>
</ul>
</li>

<li><b>Image Processing</b>
<ul>
<li>OpenCV</li>
<li>Scikit-image</li>
<li>Pillow</li>
</ul>
</li>

<li><b>Deployment</b>
<ul>
<li>Flask Web Framework</li>
</ul>
</li>

</ul>


<h2>Dataset</h2>

<p>The system is trained using publicly available dermoscopic datasets:</p>

<ul>

<li><b>HAM10000 Dataset</b> – Contains over 10,000 skin lesion images across seven classes.</li>

<li><b>ISIC Dataset</b> – A widely used dataset for skin cancer research and AI model training.</li>

</ul>


<h2>Performance Evaluation</h2>

<p>The model performance is evaluated using standard machine learning metrics:</p>

<ul>

<li>Accuracy</li>
<li>Precision</li>
<li>Recall</li>
<li>F1 Score</li>
<li>Confusion Matrix</li>
<li>ROC-AUC Curve</li>

</ul>


<h2>Impact</h2>

<p>
This hybrid AI system improves accuracy, robustness, and interpretability in skin cancer classification.
By combining multiple feature extraction techniques and machine learning models, the system provides
a reliable tool that can assist healthcare professionals in early detection and diagnosis of skin cancer.
</p>

</div>

</body>
</html>

Overwriting templates/about.html


In [91]:
%%writefile templates/index.html

<!DOCTYPE html>
<html>
<head>
    <title>FusionDerm - Hybrid Skin Cancer Classification</title>

    <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600&display=swap" rel="stylesheet">

    <link rel="stylesheet"
    href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.5.0/css/all.min.css">

    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>

<body>

<!-- NAVBAR -->
<nav class="navbar">
    <div class="logo">
        <i class="fa-solid fa-stethoscope"></i> FusionDerm
    </div>

    <div class="nav-links">
        <a href="#" onclick="showHome()">Home</a>
        <a href="#" onclick="showAbout()">About</a>
    </div>
</nav>


<!-- HOME SECTION -->
<section id="home-section">

<div class="header">
    <h1>Hybrid Skin Cancer Classification</h1>
    <p>AI Assisted Dermoscopic Lesion Analysis</p>
</div>

<div class="container">

<form method="POST" enctype="multipart/form-data">

    <div class="upload-title">
        <i class="fa-solid fa-image"></i> Upload Dermoscopic Image
    </div>

    <div class="upload-row">
        <input type="file" name="file" required>
        <button type="submit">
            <i class="fa-solid fa-microscope"></i> Predict
        </button>
    </div>

</form>

{% if img_path %}
    <img src="{{ img_path }}" class="preview">
{% endif %}


<!-- ========================= -->
<!-- ⭐ RESULT + EXPLAINABILITY -->
<!-- ========================= -->

{% if prediction %}
<div class="result-box">

    <p class="result-title">
        <i class="fa-solid fa-file-medical"></i> Diagnosis Result
    </p>

    <h2 class="result-value">{{ prediction }}</h2>

    <!-- ⭐ SCORE CAM OUTPUT -->
    {% if cam_path %}
    <div class="explain-box">
        <p class="explain-title">
            <i class="fa-solid fa-eye"></i> Lesion Attention (Score-CAM)
        </p>
        <img src="{{ cam_path }}" class="explain-img">
    </div>
    {% endif %}

    <!-- ⭐ SHAP OUTPUT -->
    {% if shap_path %}
    <div class="explain-box">
        <p class="explain-title">
            <i class="fa-solid fa-chart-bar"></i> Model Feature Importance (SHAP)
        </p>
        <img src="{{ shap_path }}" class="explain-img">
    </div>
    {% endif %}

</div>
{% endif %}

</div>
</section>



<!-- ABOUT SECTION -->
<section id="about-section" class="about-section" style="display:none;">

<h2>About FusionDerm</h2>

<p>
FusionDerm is a hybrid AI-based skin cancer classification system designed to assist
dermoscopic lesion analysis using advanced deep learning and machine learning techniques.
</p>

<ul>
<li>✔ EfficientNet extracts deep spatial image features</li>
<li>✔ Swin Transformer captures global contextual patterns</li>
<li>✔ Handcrafted texture features enhance lesion characterization</li>
<li>✔ Feature Fusion combines multiple representations</li>
<li>✔ PCA reduces dimensionality</li>
<li>✔ SMOTE balances class distribution</li>
<li>✔ XGBoost / LightGBM perform final classification</li>
</ul>

</section>


<script>
function showAbout(){
    document.getElementById("home-section").style.display="none";
    document.getElementById("about-section").style.display="block";
}
function showHome(){
    document.getElementById("home-section").style.display="block";
    document.getElementById("about-section").style.display="none";
}
</script>

<a href="/">Logout</a>
</body>
</html>


Overwriting templates/index.html


In [92]:
%%writefile static/style.css

/* ============================= */
/* GLOBAL PAGE BACKGROUND */
/* ============================= */

.page-bg{

margin:0;
padding:0;
min-height:100vh;

background:
linear-gradient(
rgba(0,0,0,0.25),
rgba(0,0,0,0.25)
),
url("https://t4.ftcdn.net/jpg/03/01/34/77/240_F_301347734_BNlV5crnfo0AYkZUXxgOC6t2qFLVLs9N.jpg");

background-size:cover;
background-position:center;
background-repeat:no-repeat;
background-attachment:fixed;

font-family:'Poppins',sans-serif;

}


/* ============================= */
/* GLOBAL HEADER COLORS */
/* ============================= */

h1, h2, h3{
color:#000;
}


/* ============================= */
/* LOGIN PAGE BACKGROUND */
/* ============================= */

.login-body{

background-image:
url("https://raw.githubusercontent.com/Ramyasri-14/skin-cancer-/main/staticlogin_bg.jpg.png");

background-size:cover;
background-position:center;
background-repeat:no-repeat;
background-attachment:fixed;

height:100vh;

display:flex;
align-items:center;
justify-content:flex-start;

padding-left:10%;

}


/* ============================= */
/* NAVBAR */
/* ============================= */

.navbar{
display:flex;
justify-content:space-between;
align-items:center;
padding:18px 40px;
color:black;
}

.logo{
font-size:22px;
font-weight:600;
color:black;
}

.nav-links a{
margin-left:25px;
color:black;
text-decoration:none;
font-size:16px;
transition:0.3s;
}

.nav-links a:hover{
opacity:0.7;
}


/* ============================= */
/* HEADER */
/* ============================= */

.header{
text-align:center;
margin-top:40px;
color:black;
}

.header h1{
font-size:42px;
margin-bottom:10px;
font-weight:600;
}

.header p{
font-size:18px;
opacity:0.9;
color:black;
}


/* ============================= */
/* GLASS CARD */
/* ============================= */

.container{
width:40%;
margin:40px auto;
padding:40px;

background:rgba(255,255,255,0.25);
backdrop-filter:blur(14px);

border-radius:20px;

box-shadow:0 10px 30px rgba(0,0,0,0.15);
}


/* ============================= */
/* UPLOAD SECTION */
/* ============================= */

.upload-title{
font-size:18px;
margin-bottom:15px;
}

.upload-row{
display:flex;
justify-content:center;
align-items:center;
gap:15px;
flex-wrap:nowrap;
}

input[type="file"]{
margin:0;
}


/* ============================= */
/* BUTTON */
/* ============================= */

button{
padding:12px 26px;
background:#3b6fc4;
color:white;
border:none;
border-radius:8px;
font-size:16px;
cursor:pointer;
transition:0.3s;
}

button:hover{
background:#2f5ca5;
transform:translateY(-2px);
box-shadow:0 5px 12px rgba(0,0,0,0.2);
}


/* ============================= */
/* PREVIEW IMAGE */
/* ============================= */

.preview{
display:block;
width:260px;
margin:25px auto;
border-radius:12px;
box-shadow:0 4px 15px rgba(0,0,0,0.2);
}


/* ============================= */
/* RESULT BOX */
/* ============================= */

.result-box{
margin-top:25px;
padding:25px;
border-radius:14px;

background:rgba(255,255,255,0.35);
box-shadow:0 0 15px rgba(0,0,0,0.15);

text-align:center;
}

.result-title{
font-size:14px;
opacity:0.8;
}

.result-value{
font-size:32px;
font-weight:700;
margin-top:8px;
letter-spacing:1px;
}


/* ============================= */
/* ABOUT SECTION */
/* ============================= */

.about-section{
width:60%;
margin:60px auto;
padding:40px;

background:rgba(255,255,255,0.3);
backdrop-filter:blur(10px);

border-radius:18px;
color:#0a2540;
}

.about-section h2{
text-align:center;
margin-bottom:20px;
}

html{
scroll-behavior:smooth;
}


/* ============================= */
/* LOGIN CARD */
/* ============================= */

.login-container{

background:rgba(255,255,255,0.9);
padding:40px;
border-radius:15px;

width:350px;

text-align:center;

box-shadow:0 8px 25px rgba(0,0,0,0.3);

}

.login-container input{

width:100%;
padding:12px;
margin:10px 0;

border-radius:8px;
border:1px solid #ccc;

}

.login-container button{

width:100%;
padding:12px;

background:#3b6fc4;
color:white;

border:none;
border-radius:8px;

font-size:16px;
cursor:pointer;

}

.login-container button:hover{

background:#2f5ca5;

}


/* ============================= */
/* AWARENESS SECTION */
/* ============================= */

.awareness{

width:70%;
margin:50px auto;
background:rgba(255,255,255,0.3);
padding:40px;
border-radius:20px;
text-align:center;

}

.awareness-img{
width:250px;
height:auto;
border-radius:12px;
margin:20px auto;
display:block;
}


/* ============================= */
/* SHAP IMAGE FIX */
/* ============================= */

.shap-img{

max-width:100%;
height:auto;

display:block;
margin:20px auto;

border-radius:10px;

}


/* ============================= */
/* PRECAUTIONS PAGE BACKGROUND */
/* ============================= */

.precautions-body{

margin:0;
padding:0;
min-height:100vh;

background:
linear-gradient(
rgba(0,0,0,0.35),
rgba(0,0,0,0.35)
),
url("https://t4.ftcdn.net/jpg/18/94/19/23/240_F_1894192325_L8oXdwHAOkXoqVctLSfUHvN7Sjq2XyKA.jpg");

background-size:cover;
background-position:center;
background-repeat:no-repeat;

font-family:'Poppins',sans-serif;

color:white;

}


/* ============================= */
/* PRECAUTIONS PAGE LAYOUT */
/* ============================= */

.precautions-container{

width:60%;
margin:120px auto;
padding:50px;

background:rgba(255,255,255,0.25);
backdrop-filter:blur(10px);

border-radius:20px;

box-shadow:0 10px 30px rgba(0,0,0,0.25);

text-align:left;

color:white;

}


/* heading */

.precautions-container h1{

font-size:42px;
margin-bottom:25px;

}


/* list styling */

.precautions-container ul{

font-size:20px;
line-height:1.8;

padding-left:20px;

}


/* back button */

.back-btn{

display:inline-block;
margin-top:30px;
padding:12px 25px;

background:#3b6fc4;
color:white;

text-decoration:none;
border-radius:8px;

font-size:16px;

transition:0.3s;

}

.back-btn:hover{

background:#2f5ca5;

}


/* ============================= */
/* LONG HORIZONTAL CARD LAYOUT */
/* ============================= */

.types{
width:100%;
margin-top:80px;
text-align:center;
}

.section{
width:100%;
margin:110px 0;
}

.card-row{

display:flex;
align-items:center;

gap:15px;

width:70%;
max-width:1000px;

background:rgba(255,255,255,0.35);
backdrop-filter:blur(10px);

padding:25px 40px;

border-radius:20px;

box-shadow:0 10px 25px rgba(0,0,0,0.2);

}

.card-row.reverse{
flex-direction:row-reverse;
}

.card-row img{

width:220px;
height:150px;

object-fit:cover;

border-radius:15px;

box-shadow:0 6px 15px rgba(0,0,0,0.3);

}

.card-text{

flex:1;
text-align:left;
max-width:650px;

}

.card-text h3{
margin-bottom:10px;
}

.card-text ul{
margin-top:10px;
padding-left:20px;
}

/* alternate card alignment */

.section:nth-child(odd) .card-row{
margin-left:40px;
margin-right:auto;
}

.section:nth-child(even) .card-row{
margin-right:40px;
margin-left:auto;
}

.cancer-desc{

margin-top:10px;

font-size:16px;

line-height:1.6;

color:#333;

max-width:500px;

margin-left:auto;
margin-right:auto;

}

.ai-text{

margin-top:10px;

font-size:15px;

line-height:1.6;

color:#333;

max-width:600px;

margin-left:auto;
margin-right:auto;

text-align:center;

}

.preview-img{

width:250px;

display:none;

margin-top:20px;

border-radius:12px;

box-shadow:0 4px 10px rgba(0,0,0,0.2);

}

.disclaimer{

margin-top:25px;

padding:15px;

background:#ffe9e9;

border-left:5px solid red;

border-radius:8px;

font-size:14px;

line-height:1.6;

max-width:600px;

margin-left:auto;
margin-right:auto;

}

.loader{

display:none;

border:6px solid #f3f3f3;

border-top:6px solid #3b6fc4;

border-radius:50%;

width:40px;
height:40px;

animation:spin 1s linear infinite;

margin:20px auto;

}

@keyframes spin{

0% {transform:rotate(0deg);}
100% {transform:rotate(360deg);}

}

.loading-text{

display:none;

text-align:center;

margin-top:10px;

font-size:15px;

color:#333;

}

Overwriting static/style.css


In [93]:
!pkill -f flask || echo "No flask running"
!pkill -f ngrok || echo "No ngrok running"



^C
^C


In [94]:
# 🔎 List processes using port 8000
!lsof -i :5000



COMMAND   PID USER   FD   TYPE DEVICE SIZE/OFF NODE NAME
python3 31310 root   46u  IPv4 466535      0t0  TCP *:5000 (LISTEN)


In [95]:
# ❌ Kill process by PID (replace 12345 with actual PID from previous cell)
!kill -9 31310


In [96]:
# ============================
# ✅ Run Flask in background
# ============================
!nohup python app.py > flask.log 2>&1 &


In [117]:

!tail -n 50 flask.log

2026-03-07 06:57:43.125617: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772866663.147983   32895 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772866663.155384   32895 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772866663.175349   32895 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772866663.175384   32895 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772866663.175387   32895 computation_placer.cc:177] computation placer alr

In [118]:
# ============================
# ✅ Start ngrok tunnel
# ============================
from pyngrok import ngrok, conf

conf.get_default().auth_token = "39VZsUg3uHwUZ2kezN5CAzoXjEs_73437ghMuVKE88G1Uwrxr"   # 🔑 put your token here
public_url = ngrok.connect(5000)
print("🌍 Public URL:", public_url)

🌍 Public URL: NgrokTunnel: "https://gleamless-unartful-dylan.ngrok-free.dev" -> "http://localhost:5000"
